
# 05 — Feature Engineering

## Project: India Air Quality Intelligence

### Objective
This notebook focuses on creating meaningful temporal, categorical, and historical features from the transformed air quality dataset.

These features will support:
- Time-based air quality analysis.
- Station-level and city-level pattern analysis.
- Lag-based and rolling-window analysis.
- Future machine learning experiments.

### Feature Engineering Strategy
1. Load and validate the analytical dataset.
2. Create temporal features.
3. Encode cyclical time patterns.
4. Investigate station-level time continuity.
5. Create gap-aware historical lag features.
6. Create rolling statistical features.
7. Validate engineered features and export the processed dataset.

### Important Considerations
- Missing and invalid AQI values will not be treated as zero.
- Station identity will use State, City, and Station Name together.
- Historical features must not use future observations.
- Features used for prediction must avoid target leakage.


In [3]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

# Reproducibility
RANDOM_STATE = 42

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Load the transformed analytical dataset
data_path = "../data/processed/air_quality_analytical.parquet"

df = pd.read_parquet(data_path)

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")

Dataset loaded successfully!
Dataset shape: (13509072, 11)


In [5]:
# Preview records
display(df.head())

# Column information
df.info()

# Check date range
print("Date range:")
print("Start:", df["Timestamp"].min())
print("End:", df["Timestamp"].max())

# Check missing values
display(df.isnull().sum())

,Station ID,State,City,Station Name,Date,Hour,AQI,Timestamp,Station,AQI_Status,AQI_Valid
0,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-11,00:00:00,NaN,2017-12-11,Andhra Pradesh | Amaravati | Secretariat Amara...,missing,NaN
1,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-12,00:00:00,263.00,2017-12-12,Andhra Pradesh | Amaravati | Secretariat Amara...,valid,263.00
2,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-13,00:00:00,137.00,2017-12-13,Andhra Pradesh | Amaravati | Secretariat Amara...,valid,137.00
3,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-14,00:00:00,85.00,2017-12-14,Andhra Pradesh | Amaravati | Secretariat Amara...,valid,85.00
4,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-15,00:00:00,127.00,2017-12-15,Andhra Pradesh | Amaravati | Secretariat Amara...,valid,127.00


<class 'pandas.DataFrame'>
RangeIndex: 13509072 entries, 0 to 13509071
Data columns (total 11 columns):
 #   Column        Dtype         
---  ------        -----         
 0   Station ID    str           
 1   State         str           
 2   City          str           
 3   Station Name  str           
 4   Date          datetime64[us]
 5   Hour          str           
 6   AQI           float64       
 7   Timestamp     datetime64[us]
 8   Station       str           
 9   AQI_Status    str           
 10  AQI_Valid     float64       
dtypes: datetime64[us](2), float64(2), str(7)
memory usage: 2.6 GB
Date range:
Start: 2017-01-01 00:00:00
End: 2023-12-31 23:00:00


Station ID            0
State                 0
City                  0
Station Name          0
Date                  0
Hour                  0
AQI             1035531
Timestamp             0
Station               0
AQI_Status            0
AQI_Valid       1035587
dtype: int64

In [6]:
# Confirm required columns exist
required_columns = [
    "State",
    "City",
    "Station Name",
    "Date",
    "Hour",
    "Timestamp",
    "AQI",
    "AQI_Valid"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Ensure timestamps are datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

# Validate AQI values
print("Total records:", len(df))
print("Valid AQI observations:", df["AQI_Valid"].notna().sum())
print("Missing/invalid AQI observations:", df["AQI_Valid"].isna().sum())
print("Duplicate timestamps within station identity:",
      df.duplicated(
          subset=["State", "City", "Station Name", "Timestamp"]
      ).sum())

print("\nValidation completed.")

Total records: 13509072
Valid AQI observations: 12473485
Missing/invalid AQI observations: 1035587
Duplicate timestamps within station identity: 0

Validation completed.



## 1. Temporal Feature Engineering

We extract meaningful time-based features from the Timestamp column.

Features:
- Year
- Month
- Day
- Day of Year
- Hour
- Day of Week
- Weekend indicator
- Quarter
- Season

These features will help identify daily, weekly, monthly and seasonal pollution patterns.


In [7]:
# Extract basic temporal features

df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Day"] = df["Timestamp"].dt.day
df["DayOfYear"] = df["Timestamp"].dt.dayofyear

df["Hour_Number"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
df["Quarter"] = df["Timestamp"].dt.quarter

# Weekend indicator: Monday=0, Sunday=6
df["Is_Weekend"] = (
    df["DayOfWeek"].isin([5, 6]).astype("int8")
)

# Add weekday names
df["Day_Name"] = df["Timestamp"].dt.day_name()

print("Basic temporal features created!")

C:\Users\khush\AppData\Local\Temp\ipykernel_16744\2738219455.py:6: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df["DayOfYear"] = df["Timestamp"].dt.dayofyear
C:\Users\khush\AppData\Local\Temp\ipykernel_16744\2738219455.py:9: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["DayOfWeek"] = df["Timestamp"].dt.dayofweek


Basic temporal features created!


In [8]:
# Define seasons
season_mapping = {
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Summer",
    4: "Summer",
    5: "Summer",
    6: "Monsoon",
    7: "Monsoon",
    8: "Monsoon",
    9: "Monsoon",
    10: "Post-Monsoon",
    11: "Post-Monsoon"
}

df["Season"] = df["Month"].map(season_mapping)

print("Seasonal feature created!")

Seasonal feature created!


In [9]:
# Cyclical encoding for hour (24-hour cycle)
df["Hour_Sin"] = np.sin(
    2 * np.pi * df["Hour_Number"] / 24
)

df["Hour_Cos"] = np.cos(
    2 * np.pi * df["Hour_Number"] / 24
)

# Cyclical encoding for month (12-month cycle)
df["Month_Sin"] = np.sin(
    2 * np.pi * (df["Month"] - 1) / 12
)

df["Month_Cos"] = np.cos(
    2 * np.pi * (df["Month"] - 1) / 12
)

# Cyclical encoding for day of week (7-day cycle)
df["DayOfWeek_Sin"] = np.sin(
    2 * np.pi * df["DayOfWeek"] / 7
)

df["DayOfWeek_Cos"] = np.cos(
    2 * np.pi * df["DayOfWeek"] / 7
)

print("Cyclical features created!")

Cyclical features created!


In [10]:
# Inspect newly engineered features

temporal_features = [
    "Timestamp",
    "Year",
    "Month",
    "Day",
    "DayOfYear",
    "Hour_Number",
    "DayOfWeek",
    "Day_Name",
    "Quarter",
    "Is_Weekend",
    "Season",
    "Hour_Sin",
    "Hour_Cos",
    "Month_Sin",
    "Month_Cos",
    "DayOfWeek_Sin",
    "DayOfWeek_Cos"
]

display(df[temporal_features].head(10))

print("Total rows:", len(df))
print("Missing seasonal values:", df["Season"].isna().sum())
print("Missing hour values:", df["Hour_Number"].isna().sum())
print("Unique years:", sorted(df["Year"].unique()))
print("Unique seasons:", df["Season"].unique())

,Timestamp,Year,Month,Day,DayOfYear,Hour_Number,DayOfWeek,Day_Name,Quarter,Is_Weekend,Season,Hour_Sin,Hour_Cos,Month_Sin,Month_Cos,DayOfWeek_Sin,DayOfWeek_Cos
0,2017-12-11,2017,12,11,345,0,0,Monday,4,0,Winter,0.00,1.00,-0.50,0.87,0.00,1.00
1,2017-12-12,2017,12,12,346,0,1,Tuesday,4,0,Winter,0.00,1.00,-0.50,0.87,0.78,0.62
2,2017-12-13,2017,12,13,347,0,2,Wednesday,4,0,Winter,0.00,1.00,-0.50,0.87,0.97,-0.22
3,2017-12-14,2017,12,14,348,0,3,Thursday,4,0,Winter,0.00,1.00,-0.50,0.87,0.43,-0.90
4,2017-12-15,2017,12,15,349,0,4,Friday,4,0,Winter,0.00,1.00,-0.50,0.87,-0.43,-0.90
5,2017-12-16,2017,12,16,350,0,5,Saturday,4,1,Winter,0.00,1.00,-0.50,0.87,-0.97,-0.22
6,2017-12-17,2017,12,17,351,0,6,Sunday,4,1,Winter,0.00,1.00,-0.50,0.87,-0.78,0.62
7,2017-12-18,2017,12,18,352,0,0,Monday,4,0,Winter,0.00,1.00,-0.50,0.87,0.00,1.00
8,2017-12-19,2017,12,19,353,0,1,Tuesday,4,0,Winter,0.00,1.00,-0.50,0.87,0.78,0.62
9,2017-12-20,2017,12,20,354,0,2,Wednesday,4,0,Winter,0.00,1.00,-0.50,0.87,0.97,-0.22


Total rows: 13509072
Missing seasonal values: 0
Missing hour values: 0
Unique years: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
Unique seasons: <ArrowStringArray>
['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
Length: 4, dtype: str



## 2. Station-Level Time Continuity Analysis

Before creating historical features, we investigate the temporal coverage of individual monitoring stations.

This step helps identify:
- Number of unique monitoring stations.
- Number of observations per station.
- Whether records are already chronologically ordered.
- Gaps between consecutive timestamps.
- Missing hourly readings.

This is important because lag and rolling features must represent genuine historical information without accidentally treating distant observations as consecutive hours.


In [11]:
# Count unique stations using the composite station identity
station_columns = ["State", "City", "Station Name"]

print(
    "Unique station identities:",
    df[station_columns].drop_duplicates().shape[0]
)

# Number of observations per station
station_counts = (
    df.groupby(station_columns, observed=True)
      .size()
      .reset_index(name="Total_Records")
      .sort_values("Total_Records", ascending=False)
)

display(station_counts.head(10))

print("\nStation record-count summary:")
display(station_counts["Total_Records"].describe())

Unique station identities: 529


,State,City,Station Name,Total_Records
100,Delhi,Delhi,Shadipur Delhi CPCB,59568
88,Delhi,Delhi,NSIT Dwarka Delhi CPCB,59448
86,Delhi,Delhi,Mandir Marg Delhi DPCC,59304
75,Delhi,Delhi,DTU Delhi CPCB,59256
79,Delhi,Delhi,IHBAS Dilshad Garden Delhi CPCB,59184
101,Delhi,Delhi,Sirifort Delhi CPCB,58488
10,Andhra Pradesh,Visakhapatnam,GVM Corporation Visakhapatnam APPCB,58392
80,Delhi,Delhi,ITO Delhi CPCB,58224
508,Uttar Pradesh,Varanasi,Ardhali Bazar Varanasi UPPCB,57960
98,Delhi,Delhi,R K Puram Delhi DPCC,57552



Station record-count summary:


count     529.00
mean    25537.00
std     18189.46
min        96.00
25%      8112.00
50%     21528.00
75%     42672.00
max     59568.00
Name: Total_Records, dtype: float64

In [12]:
# Check chronological ordering within each station
is_sorted_by_station_time = (
    df.groupby("Station", sort=False)["Timestamp"]
      .apply(lambda x: x.is_monotonic_increasing)
)

print("Stations checked:", len(is_sorted_by_station_time))
print("Stations already chronologically sorted:",
      is_sorted_by_station_time.sum())
print("Stations not chronologically sorted:",
      (~is_sorted_by_station_time).sum())

Stations checked: 529
Stations already chronologically sorted: 0
Stations not chronologically sorted: 529


In [13]:
# Select the 5 stations with the most records
top_stations = station_counts.head(5)

for _, station_row in top_stations.iterrows():

    mask = (
        (df["State"] == station_row["State"]) &
        (df["City"] == station_row["City"]) &
        (df["Station Name"] == station_row["Station Name"])
    )

    station_sample = (
        df.loc[mask, ["Timestamp", "AQI_Valid"]]
          .sort_values("Timestamp")
          .head(1000)
          .copy()
    )

    station_sample["Time_Gap"] = (
        station_sample["Timestamp"].diff()
    )

    print(
        f"\n{station_row['City']} — "
        f"{station_row['Station Name']}"
    )

    display(
        station_sample["Time_Gap"]
        .value_counts()
        .head(10)
    )


Delhi — Shadipur Delhi CPCB


Time_Gap
0 days 01:00:00    998
1 days 01:00:00      1
Name: count, dtype: int64


Delhi — NSIT Dwarka Delhi CPCB


Time_Gap
0 days 01:00:00    998
7 days 01:00:00      1
Name: count, dtype: int64


Delhi — Mandir Marg Delhi DPCC


Time_Gap
0 days 01:00:00    998
5 days 01:00:00      1
Name: count, dtype: int64


Delhi — DTU Delhi CPCB


Time_Gap
0 days 01:00:00    999
Name: count, dtype: int64


Delhi — IHBAS Dilshad Garden Delhi CPCB


Time_Gap
0 days 01:00:00    999
Name: count, dtype: int64


## 3. Chronological Sorting

The original dataset is not chronologically sorted within station groups.

We sort observations using the composite station identity and timestamp. This ensures that historical features are calculated in the correct temporal order.

A stable sorting algorithm is used to preserve the relative order of records with equal sorting keys.


In [14]:
import importlib.util
import os

print("DuckDB installed:",
      importlib.util.find_spec("duckdb") is not None)

try:
    import psutil

    memory = psutil.virtual_memory()

    print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
    print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
    print(f"Used RAM: {memory.percent}%")

except ImportError:
    print("psutil is not installed.")

DuckDB installed: False
Total RAM: 15.70 GB
Available RAM: 1.06 GB
Used RAM: 93.3%


In [15]:
print("DataFrame shape:", df.shape)

print("Timestamp dtype:", df["Timestamp"].dtype)

print("Valid AQI observations:", df["AQI_Valid"].notna().sum())

print("Unique stations:",
      df[["State", "City", "Station Name"]]
      .drop_duplicates().shape[0])

DataFrame shape: (13509072, 27)
Timestamp dtype: datetime64[us]
Valid AQI observations: 12473485
Unique stations: 529


In [1]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 2.0.0



## 4. Scalable Feature Engineering with DuckDB

Due to the size of the dataset, DuckDB is used for memory-efficient analytical processing.

The workflow uses:
- Direct Parquet reading.
- SQL-based chronological sorting.
- Station-level window functions.
- Timestamp-aware lag features.
- Historical rolling statistics.
- Parquet output for downstream modeling.


In [3]:
import duckdb
import os

# Project paths
PROJECT_ROOT = os.path.abspath("..")

INPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "air_quality_analytical.parquet"
)

OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "air_quality_feature_engineered.parquet"
)

TEMP_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "duckdb_temp"
)

os.makedirs(TEMP_PATH, exist_ok=True)

# Create DuckDB connection
con = duckdb.connect(
    os.path.join(PROJECT_ROOT, "data", "processed", "air_quality.duckdb")
)

# Configure memory and temporary storage
con.execute("SET memory_limit = '6GB'")
con.execute(f"SET temp_directory = '{TEMP_PATH}'")
con.execute("SET threads = 4")

print("DuckDB configured successfully!")

DuckDB configured successfully!


In [4]:
# Inspect source schema
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{INPUT_PATH}')"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,Station ID,VARCHAR,YES,None,None,None
1,State,VARCHAR,YES,None,None,None
2,City,VARCHAR,YES,None,None,None
3,Station Name,VARCHAR,YES,None,None,None
4,Date,TIMESTAMP,YES,None,None,None
5,Hour,VARCHAR,YES,None,None,None
6,AQI,DOUBLE,YES,None,None,None
7,Timestamp,TIMESTAMP,YES,None,None,None
8,Station,VARCHAR,YES,None,None,None
9,AQI_Status,VARCHAR,YES,None,None,None


In [5]:
source_stats = con.execute(f"""
    SELECT
        COUNT(*) AS total_records,
        COUNT(AQI_Valid) AS valid_aqi_records,
        COUNT(*) - COUNT(AQI_Valid) AS missing_or_invalid,
        COUNT(DISTINCT Station) AS unique_stations,
        MIN(Timestamp) AS start_timestamp,
        MAX(Timestamp) AS end_timestamp
    FROM read_parquet('{INPUT_PATH}')
""").df()

display(source_stats)

,total_records,valid_aqi_records,missing_or_invalid,unique_stations,start_timestamp,end_timestamp
0,13509072,12473485,1035587,529,2017-01-01,2023-12-31 23:00:00


In [6]:
con.execute(f"""
    CREATE OR REPLACE VIEW air_quality_source AS
    SELECT *
    FROM read_parquet('{INPUT_PATH}')
""")

print("Source Parquet registered successfully!")

Source Parquet registered successfully!


In [7]:
test_result = con.execute("""
    SELECT
        State,
        City,
        "Station Name",
        COUNT(*) AS records,
        COUNT(AQI_Valid) AS valid_records
    FROM air_quality_source
    GROUP BY State, City, "Station Name"
    ORDER BY records DESC
    LIMIT 10
""").df()

display(test_result)

,State,City,Station Name,records,valid_records
0,Delhi,Delhi,Shadipur Delhi CPCB,59568,58483
1,Delhi,Delhi,NSIT Dwarka Delhi CPCB,59448,56127
2,Delhi,Delhi,Mandir Marg Delhi DPCC,59304,53906
3,Delhi,Delhi,DTU Delhi CPCB,59256,57125
4,Delhi,Delhi,IHBAS Dilshad Garden Delhi CPCB,59184,55096
5,Delhi,Delhi,Sirifort Delhi CPCB,58488,54611
6,Andhra Pradesh,Visakhapatnam,GVM Corporation Visakhapatnam APPCB,58392,53623
7,Delhi,Delhi,ITO Delhi CPCB,58224,55230
8,Uttar Pradesh,Varanasi,Ardhali Bazar Varanasi UPPCB,57960,55044
9,Delhi,Delhi,R K Puram Delhi DPCC,57552,52916



## 5. Full-Dataset Timestamp Gap Analysis

We use the SQL LAG() window function to identify the previous timestamp for each monitoring station.

The time difference between consecutive timestamps helps us identify:
- Continuous hourly observations.
- Missing hourly records.
- Multi-hour or multi-day gaps.
- Station-level monitoring interruptions.

This analysis determines how historical lag features should be constructed.


In [9]:
gap_distribution = con.execute("""
    WITH station_history AS (
        SELECT
            State,
            City,
            "Station Name",
            Timestamp,

            LAG(Timestamp) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
            ) AS Previous_Timestamp

        FROM air_quality_source
    ),

    timestamp_gaps AS (
        SELECT
            date_diff(
                'second',
                Previous_Timestamp,
                Timestamp
            ) / 3600.0 AS Gap_Hours

        FROM station_history
        WHERE Previous_Timestamp IS NOT NULL
    )

    SELECT
        Gap_Hours,
        COUNT(*) AS Gap_Count

    FROM timestamp_gaps
    GROUP BY Gap_Hours
    ORDER BY Gap_Count DESC
    LIMIT 20
""").df()

display(gap_distribution)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Gap_Hours,Gap_Count
0,1.0,13500150
1,25.0,3932
2,49.0,1562
3,73.0,722
4,97.0,418
5,121.0,306
6,145.0,207
7,169.0,171
8,193.0,105
9,313.0,100


In [10]:
gap_summary = con.execute("""
    WITH station_history AS (
        SELECT
            State,
            City,
            "Station Name",
            Timestamp,

            LAG(Timestamp) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
            ) AS Previous_Timestamp

        FROM air_quality_source
    ),

    timestamp_gaps AS (
        SELECT
            date_diff(
                'second',
                Previous_Timestamp,
                Timestamp
            ) / 3600.0 AS Gap_Hours

        FROM station_history
        WHERE Previous_Timestamp IS NOT NULL
    )

    SELECT
        COUNT(*) AS Total_Intervals,

        COUNT(*) FILTER (
            WHERE Gap_Hours = 1
        ) AS Exact_One_Hour_Intervals,

        COUNT(*) FILTER (
            WHERE Gap_Hours > 1
        ) AS Gaps_Greater_Than_One_Hour,

        COUNT(*) FILTER (
            WHERE Gap_Hours > 24
        ) AS Gaps_Greater_Than_24_Hours,

        COUNT(*) FILTER (
            WHERE Gap_Hours > 168
        ) AS Gaps_Greater_Than_168_Hours,

        MAX(Gap_Hours) AS Maximum_Gap_Hours

    FROM timestamp_gaps
""").df()

display(gap_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Intervals,Exact_One_Hour_Intervals,Gaps_Greater_Than_One_Hour,Gaps_Greater_Than_24_Hours,Gaps_Greater_Than_168_Hours,Maximum_Gap_Hours
0,13508543,13500150,8393,8393,1246,16369.0


In [11]:
station_gap_summary = con.execute("""
    WITH station_history AS (
        SELECT
            State,
            City,
            "Station Name",
            Timestamp,

            LAG(Timestamp) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
            ) AS Previous_Timestamp

        FROM air_quality_source
    ),

    station_gaps AS (
        SELECT
            State,
            City,
            "Station Name",

            COUNT(*) FILTER (
                WHERE Previous_Timestamp IS NOT NULL
            ) AS Total_Intervals,

            COUNT(*) FILTER (
                WHERE date_diff(
                    'second',
                    Previous_Timestamp,
                    Timestamp
                ) = 3600
            ) AS One_Hour_Intervals

        FROM station_history
        GROUP BY State, City, "Station Name"
    )

    SELECT
        *,
        ROUND(
            100.0 * One_Hour_Intervals /
            NULLIF(Total_Intervals, 0),
            2
        ) AS Hourly_Continuity_Percent

    FROM station_gaps
    ORDER BY Hourly_Continuity_Percent ASC
    LIMIT 15
""").df()

display(station_gap_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,State,City,Station Name,Total_Intervals,One_Hour_Intervals,Hourly_Continuity_Percent
0,Tamil Nadu,Chengalpattu,Crescent University Chengalpattu TNPCB,6863,6795,99.01
1,Maharashtra,Boisar,Khaira Boisar MPCB,3551,3523,99.21
2,Maharashtra,Thane,Kasarvadavali Thane MPCB,3695,3666,99.22
3,Arunachal Pradesh,Naharlagun,Naharlagun Naharlagun APSPCB,17831,17694,99.23
4,Chhattisgarh,Bhilai,Civic Center Bhilai Bhilai Steel Plant,9935,9889,99.54
5,Tamil Nadu,Vellore,Vasanthapuram Vellore TNPCB,9575,9533,99.56
6,Tamil Nadu,Salem,Sona College of Technology Salem TNPCB,9527,9485,99.56
7,Karnataka,Udupi,Brahmagiri Udupi KSPCB,19487,19413,99.62
8,Gujarat,Nandesari,GIDC Nandesari Nandesari Ind. Association,26135,26039,99.63
9,Karnataka,Raichur,Mahatma Basaveswar Colony Kalaburgi KSPCB,6527,6503,99.63


In [12]:
con.execute(f"""
COPY (
    WITH lagged_data AS (
        SELECT
            *,
            LAG(Timestamp, 1) OVER station_window AS Previous_Timestamp_1H,
            LAG(AQI_Valid, 1) OVER station_window AS Previous_AQI_1H,

            LAG(Timestamp, 24) OVER station_window AS Previous_Timestamp_24H,
            LAG(AQI_Valid, 24) OVER station_window AS Previous_AQI_24H,

            LAG(Timestamp, 168) OVER station_window AS Previous_Timestamp_168H,
            LAG(AQI_Valid, 168) OVER station_window AS Previous_AQI_168H

        FROM air_quality_source

        WINDOW station_window AS (
            PARTITION BY State, City, "Station Name"
            ORDER BY Timestamp
        )
    )

    SELECT
        *,

        -- Temporal features
        YEAR(Timestamp) AS Year,
        MONTH(Timestamp) AS Month,
        DAY(Timestamp) AS Day,
        DAYOFYEAR(Timestamp) AS DayOfYear,
        HOUR(Timestamp) AS Hour_Number,
        ISODOW(Timestamp) - 1 AS DayOfWeek,
        QUARTER(Timestamp) AS Quarter,

        CASE
            WHEN ISODOW(Timestamp) IN (6, 7) THEN 1
            ELSE 0
        END AS Is_Weekend,

        STRFTIME(Timestamp, '%A') AS Day_Name,

        CASE
            WHEN MONTH(Timestamp) IN (12, 1, 2) THEN 'Winter'
            WHEN MONTH(Timestamp) IN (3, 4, 5) THEN 'Summer'
            WHEN MONTH(Timestamp) IN (6, 7, 8, 9) THEN 'Monsoon'
            ELSE 'Post-Monsoon'
        END AS Season,

        -- Cyclical temporal encoding
        SIN(2 * PI() * HOUR(Timestamp) / 24.0) AS Hour_Sin,
        COS(2 * PI() * HOUR(Timestamp) / 24.0) AS Hour_Cos,

        SIN(2 * PI() * (MONTH(Timestamp) - 1) / 12.0) AS Month_Sin,
        COS(2 * PI() * (MONTH(Timestamp) - 1) / 12.0) AS Month_Cos,

        SIN(2 * PI() * (ISODOW(Timestamp) - 1) / 7.0) AS DayOfWeek_Sin,
        COS(2 * PI() * (ISODOW(Timestamp) - 1) / 7.0) AS DayOfWeek_Cos,

        -- Exact timestamp-aligned lag features
        CASE
            WHEN date_diff('second', Previous_Timestamp_1H, Timestamp) = 3600
            THEN Previous_AQI_1H
            ELSE NULL
        END AS AQI_Lag_1H,

        CASE
            WHEN date_diff('second', Previous_Timestamp_24H, Timestamp) = 86400
            THEN Previous_AQI_24H
            ELSE NULL
        END AS AQI_Lag_24H,

        CASE
            WHEN date_diff('second', Previous_Timestamp_168H, Timestamp) = 604800
            THEN Previous_AQI_168H
            ELSE NULL
        END AS AQI_Lag_168H,

        -- Historical timestamp availability indicators
        CASE
            WHEN date_diff('second', Previous_Timestamp_1H, Timestamp) = 3600
            THEN 1 ELSE 0
        END AS Has_1H_History,

        CASE
            WHEN date_diff('second', Previous_Timestamp_24H, Timestamp) = 86400
            THEN 1 ELSE 0
        END AS Has_24H_History,

        CASE
            WHEN date_diff('second', Previous_Timestamp_168H, Timestamp) = 604800
            THEN 1 ELSE 0
        END AS Has_168H_History

    FROM lagged_data
)
TO '{OUTPUT_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Feature-engineered Parquet file created successfully!")
print("Output:", OUTPUT_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-engineered Parquet file created successfully!
Output: c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_feature_engineered.parquet


In [13]:
con.execute(f"""
CREATE OR REPLACE VIEW air_quality_features AS
SELECT *
FROM read_parquet('{OUTPUT_PATH}')
""")

validation = con.execute("""
SELECT
    COUNT(*) AS Total_Records,
    COUNT(AQI_Valid) AS Valid_AQI_Records,

    COUNT(AQI_Lag_1H) AS Available_1H_Lags,
    COUNT(AQI_Lag_24H) AS Available_24H_Lags,
    COUNT(AQI_Lag_168H) AS Available_168H_Lags,

    COUNT(*) FILTER (WHERE Has_1H_History = 1) AS Exact_1H_History,
    COUNT(*) FILTER (WHERE Has_24H_History = 1) AS Exact_24H_History,
    COUNT(*) FILTER (WHERE Has_168H_History = 1) AS Exact_168H_History,

    MIN(Year) AS First_Year,
    MAX(Year) AS Last_Year

FROM air_quality_features
""").df()

display(validation)

,Total_Records,Valid_AQI_Records,Available_1H_Lags,Available_24H_Lags,Available_168H_Lags,Exact_1H_History,Exact_24H_History,Exact_168H_History,First_Year,Last_Year
0,13509072,12473485,12472563,12365127,11480020,13500150,13294944,12260472,2017,2023


In [14]:
sample = con.execute("""
SELECT
    City,
    "Station Name",
    Timestamp,
    AQI_Valid,
    AQI_Lag_1H,
    AQI_Lag_24H,
    AQI_Lag_168H,
    Has_1H_History,
    Has_24H_History,
    Has_168H_History,
    Season
FROM air_quality_features
WHERE AQI_Valid IS NOT NULL
LIMIT 15
""").df()

display(sample)

,City,Station Name,Timestamp,AQI_Valid,AQI_Lag_1H,AQI_Lag_24H,AQI_Lag_168H,Has_1H_History,Has_24H_History,Has_168H_History,Season
0,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 13:00:00,58.0,NaN,NaN,NaN,1,0,0,Monsoon
1,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 14:00:00,69.0,58.0,NaN,NaN,1,0,0,Monsoon
2,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 15:00:00,85.0,69.0,NaN,NaN,1,0,0,Monsoon
3,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 16:00:00,95.0,85.0,NaN,NaN,1,0,0,Monsoon
4,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 17:00:00,81.0,95.0,NaN,NaN,1,0,0,Monsoon
5,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 18:00:00,76.0,81.0,NaN,NaN,1,0,0,Monsoon
6,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 19:00:00,87.0,76.0,NaN,NaN,1,0,0,Monsoon
7,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 20:00:00,98.0,87.0,NaN,NaN,1,0,0,Monsoon
8,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 21:00:00,102.0,98.0,NaN,NaN,1,0,0,Monsoon
9,Bihar Sharif,D M Colony Bihar Sharif BSPCB,2021-07-08 22:00:00,106.0,102.0,NaN,NaN,1,0,0,Monsoon


In [15]:
ROLLING_OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "air_quality_feature_engineered_v2.parquet"
)

con.execute(f"""
COPY (
    WITH rolling_features AS (
        SELECT
            *,

            COUNT(AQI_Valid) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
                RANGE BETWEEN INTERVAL '24 hours' PRECEDING
                AND INTERVAL '1 hour' PRECEDING
            ) AS AQI_Rolling_Count_24H,

            AVG(AQI_Valid) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
                RANGE BETWEEN INTERVAL '24 hours' PRECEDING
                AND INTERVAL '1 hour' PRECEDING
            ) AS Rolling_Mean_24H_Raw,

            COUNT(AQI_Valid) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
                RANGE BETWEEN INTERVAL '168 hours' PRECEDING
                AND INTERVAL '1 hour' PRECEDING
            ) AS AQI_Rolling_Count_7D,

            AVG(AQI_Valid) OVER (
                PARTITION BY State, City, "Station Name"
                ORDER BY Timestamp
                RANGE BETWEEN INTERVAL '168 hours' PRECEDING
                AND INTERVAL '1 hour' PRECEDING
            ) AS Rolling_Mean_7D_Raw

        FROM air_quality_features
    )

    SELECT
        * EXCLUDE (Rolling_Mean_24H_Raw, Rolling_Mean_7D_Raw),

        CASE
            WHEN AQI_Rolling_Count_24H >= 18
            THEN Rolling_Mean_24H_Raw
            ELSE NULL
        END AS AQI_Rolling_Mean_24H,

        CASE
            WHEN AQI_Rolling_Count_7D >= 126
            THEN Rolling_Mean_7D_Raw
            ELSE NULL
        END AS AQI_Rolling_Mean_7D

    FROM rolling_features
)
TO '{ROLLING_OUTPUT_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Rolling-window features created successfully!")
print("Output:", ROLLING_OUTPUT_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rolling-window features created successfully!
Output: c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_feature_engineered_v2.parquet


In [16]:
con.execute(f"""
CREATE OR REPLACE VIEW air_quality_features_v2 AS
SELECT *
FROM read_parquet('{ROLLING_OUTPUT_PATH}')
""")

rolling_validation = con.execute("""
SELECT
    COUNT(*) AS Total_Records,

    COUNT(AQI_Rolling_Mean_24H) AS Available_24H_Rolling_Means,
    COUNT(AQI_Rolling_Mean_7D) AS Available_7D_Rolling_Means,

    COUNT(*) FILTER (
        WHERE AQI_Rolling_Count_24H >= 18
    ) AS Windows_Meeting_24H_Threshold,

    COUNT(*) FILTER (
        WHERE AQI_Rolling_Count_7D >= 126
    ) AS Windows_Meeting_7D_Threshold,

    MIN(AQI_Rolling_Mean_24H) AS Minimum_24H_Mean,
    MAX(AQI_Rolling_Mean_24H) AS Maximum_24H_Mean,

    MIN(AQI_Rolling_Mean_7D) AS Minimum_7D_Mean,
    MAX(AQI_Rolling_Mean_7D) AS Maximum_7D_Mean

FROM air_quality_features_v2
""").df()

display(rolling_validation)

,Total_Records,Available_24H_Rolling_Means,Available_7D_Rolling_Means,Windows_Meeting_24H_Threshold,Windows_Meeting_7D_Threshold,Minimum_24H_Mean,Maximum_24H_Mean,Minimum_7D_Mean,Maximum_7D_Mean
0,13509072,12157246,11932485,12157246,11932485,1.0,500.0,1.464567,500.0


In [17]:
EXACT_LAG_OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "air_quality_feature_engineered_final.parquet"
)

con.execute(f"""
COPY (
    SELECT
        f.* EXCLUDE (
            AQI_Lag_1H,
            AQI_Lag_24H,
            AQI_Lag_168H,
            Has_1H_History,
            Has_24H_History,
            Has_168H_History
        ),

        -- Exact historical AQI values
        p1.AQI_Valid AS AQI_Lag_1H,
        p24.AQI_Valid AS AQI_Lag_24H,
        p168.AQI_Valid AS AQI_Lag_168H,

        -- Exact timestamp availability flags
        CASE WHEN p1.Timestamp IS NOT NULL THEN 1 ELSE 0 END
            AS Has_1H_History,

        CASE WHEN p24.Timestamp IS NOT NULL THEN 1 ELSE 0 END
            AS Has_24H_History,

        CASE WHEN p168.Timestamp IS NOT NULL THEN 1 ELSE 0 END
            AS Has_168H_History

    FROM air_quality_features_v2 f

    LEFT JOIN air_quality_features_v2 p1
        ON f.State = p1.State
        AND f.City = p1.City
        AND f."Station Name" = p1."Station Name"
        AND p1.Timestamp = f.Timestamp - INTERVAL '1 hour'

    LEFT JOIN air_quality_features_v2 p24
        ON f.State = p24.State
        AND f.City = p24.City
        AND f."Station Name" = p24."Station Name"
        AND p24.Timestamp = f.Timestamp - INTERVAL '24 hours'

    LEFT JOIN air_quality_features_v2 p168
        ON f.State = p168.State
        AND f.City = p168.City
        AND f."Station Name" = p168."Station Name"
        AND p168.Timestamp = f.Timestamp - INTERVAL '168 hours'
)
TO '{EXACT_LAG_OUTPUT_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Corrected exact-time lag features generated!")
print(EXACT_LAG_OUTPUT_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Corrected exact-time lag features generated!
c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_feature_engineered_final.parquet


In [18]:
con.execute(f"""
CREATE OR REPLACE VIEW air_quality_features_final AS
SELECT *
FROM read_parquet('{EXACT_LAG_OUTPUT_PATH}')
""")

exact_lag_validation = con.execute("""
SELECT
    COUNT(*) AS Total_Records,

    COUNT(*) FILTER (WHERE Has_1H_History = 1)
        AS Exact_1H_Timestamps,

    COUNT(*) FILTER (WHERE Has_24H_History = 1)
        AS Exact_24H_Timestamps,

    COUNT(*) FILTER (WHERE Has_168H_History = 1)
        AS Exact_168H_Timestamps,

    COUNT(AQI_Lag_1H) AS Valid_1H_Lag_Values,
    COUNT(AQI_Lag_24H) AS Valid_24H_Lag_Values,
    COUNT(AQI_Lag_168H) AS Valid_168H_Lag_Values

FROM air_quality_features_final
""").df()

display(exact_lag_validation)

,Total_Records,Exact_1H_Timestamps,Exact_24H_Timestamps,Exact_168H_Timestamps,Valid_1H_Lag_Values,Valid_24H_Lag_Values,Valid_168H_Lag_Values
0,13509072,13500150,13294944,12966576,12472563,12365127,12019379


In [20]:
final_quality_check = con.execute("""
SELECT
    COUNT(*) AS Total_Records,

    COUNT(DISTINCT State || '|' || City || '|' ||
          "Station Name" || '|' || CAST(Timestamp AS VARCHAR))
        AS Unique_Station_Timestamps,

    COUNT(*) FILTER (
        WHERE Year NOT BETWEEN 2017 AND 2023
    ) AS Invalid_Years,

    COUNT(*) FILTER (
        WHERE Month NOT BETWEEN 1 AND 12
           OR Day NOT BETWEEN 1 AND 31
           OR Hour_Number NOT BETWEEN 0 AND 23
           OR DayOfWeek NOT BETWEEN 0 AND 6
    ) AS Invalid_Temporal_Values,

    COUNT(*) FILTER (
        WHERE Season NOT IN (
            'Winter', 'Summer', 'Monsoon', 'Post-Monsoon'
        )
    ) AS Invalid_Seasons,

    COUNT(*) FILTER (
        WHERE ABS(Hour_Sin) > 1.000001
           OR ABS(Hour_Cos) > 1.000001
           OR ABS(Month_Sin) > 1.000001
           OR ABS(Month_Cos) > 1.000001
           OR ABS(DayOfWeek_Sin) > 1.000001
           OR ABS(DayOfWeek_Cos) > 1.000001
    ) AS Invalid_Cyclical_Values,

    COUNT(*) FILTER (
        WHERE AQI_Lag_1H NOT BETWEEN 1 AND 500
           OR AQI_Lag_24H NOT BETWEEN 1 AND 500
           OR AQI_Lag_168H NOT BETWEEN 1 AND 500
    ) AS Invalid_Lag_Values,

    COUNT(*) FILTER (
        WHERE AQI_Rolling_Mean_24H NOT BETWEEN 1 AND 500
           OR AQI_Rolling_Mean_7D NOT BETWEEN 1 AND 500
    ) AS Invalid_Rolling_Means,

    COUNT(*) FILTER (
        WHERE AQI_Rolling_Mean_24H IS NOT NULL
          AND AQI_Rolling_Count_24H < 18
    ) AS Invalid_24H_Rolling_Thresholds,

    COUNT(*) FILTER (
        WHERE AQI_Rolling_Mean_7D IS NOT NULL
          AND AQI_Rolling_Count_7D < 126
    ) AS Invalid_7D_Rolling_Thresholds

FROM air_quality_features_final
""").df()

display(final_quality_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Records,Unique_Station_Timestamps,Invalid_Years,Invalid_Temporal_Values,Invalid_Seasons,Invalid_Cyclical_Values,Invalid_Lag_Values,Invalid_Rolling_Means,Invalid_24H_Rolling_Thresholds,Invalid_7D_Rolling_Thresholds
0,13509072,13509072,0,0,0,0,0,0,0,0


In [21]:
final_schema = con.execute("""
DESCRIBE air_quality_features_final
""").df()

display(final_schema)

,column_name,column_type,null,key,default,extra
0,Station ID,VARCHAR,YES,None,None,None
1,State,VARCHAR,YES,None,None,None
2,City,VARCHAR,YES,None,None,None
3,Station Name,VARCHAR,YES,None,None,None
4,Date,TIMESTAMP,YES,None,None,None
5,Hour,VARCHAR,YES,None,None,None
6,AQI,DOUBLE,YES,None,None,None
7,Timestamp,TIMESTAMP,YES,None,None,None
8,Station,VARCHAR,YES,None,None,None
9,AQI_Status,VARCHAR,YES,None,None,None


In [22]:
file_size_mb = os.path.getsize(EXACT_LAG_OUTPUT_PATH) / (1024 ** 2)

print(f"Final Parquet file size: {file_size_mb:.2f} MB")
print(f"Final file: {EXACT_LAG_OUTPUT_PATH}")

Final Parquet file size: 378.10 MB
Final file: c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_feature_engineered_final.parquet


In [23]:
CLEAN_OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "air_quality_model_ready.parquet"
)

con.execute(f"""
COPY (
    SELECT * EXCLUDE (
        Previous_Timestamp_1H,
        Previous_AQI_1H,
        Previous_Timestamp_24H,
        Previous_AQI_24H,
        Previous_Timestamp_168H,
        Previous_AQI_168H
    )
    FROM air_quality_features_final
)
TO '{CLEAN_OUTPUT_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Clean model-ready dataset saved!")
print("Path:", CLEAN_OUTPUT_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clean model-ready dataset saved!
Path: c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_model_ready.parquet


In [24]:
con.execute(f"""
CREATE OR REPLACE VIEW air_quality_model_ready AS
SELECT *
FROM read_parquet('{CLEAN_OUTPUT_PATH}')
""")

final_check = con.execute("""
SELECT
    COUNT(*) AS Total_Records,
    COUNT(DISTINCT State || '|' || City || '|' ||
          "Station Name" || '|' || CAST(Timestamp AS VARCHAR))
        AS Unique_Station_Timestamps,
    COUNT(AQI_Valid) AS Valid_AQI_Records,
    COUNT(AQI_Lag_1H) AS Valid_1H_Lags,
    COUNT(AQI_Lag_24H) AS Valid_24H_Lags,
    COUNT(AQI_Lag_168H) AS Valid_168H_Lags,
    COUNT(AQI_Rolling_Mean_24H) AS Valid_24H_Rolling_Means,
    COUNT(AQI_Rolling_Mean_7D) AS Valid_7D_Rolling_Means
FROM air_quality_model_ready
""").df()

display(final_check)

print("Final columns:", len(con.execute(
    "DESCRIBE air_quality_model_ready"
).df()))

print(
    "Final file size:",
    round(os.path.getsize(CLEAN_OUTPUT_PATH) / (1024 ** 2), 2),
    "MB"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Total_Records,Unique_Station_Timestamps,Valid_AQI_Records,Valid_1H_Lags,Valid_24H_Lags,Valid_168H_Lags,Valid_24H_Rolling_Means,Valid_7D_Rolling_Means
0,13509072,13509072,12473485,12472563,12365127,12019379,12157246,11932485


Final columns: 37
Final file size: 251.98 MB
